<div style="width: 100%; text-align: center; margin-bottom: 20px;">
  <img src="../elementos/cabecalho_branco.jpg" alt="Cabeçalho" style="max-width: 100%; height: auto;">
</div>

# Testando Hipóteses

**Autor**: Carlos Gabriel de Oliveira Campos

**Professor**: Dr. Daniel Roberto Cassar

Monstro: Devorador de mentes

---

### Função de suporte

Disponibilizada em [1].

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

def conclusao(pvalor, significancia):

    print("p-valor =", pvalor)

    print()

    if pvalor < significancia:
        print("Conclusão do teste de hipótese: \033[31mdevemos rejeitar H0\033[0m")
    else:
        print("Conclusão do teste de hipótese: \x1b[32mnão devemos rejeitar H0\x1b[0m")

### Importação dos dados do Censo Ilum

In [2]:
df_original = pd.read_excel("Censo Ilum.xlsx")

### Definição de nível de confinaça

Como padrão na ciência, será definido um nível de confiança de 95%, o que implica uma significância de 5%.

In [3]:
NIVEL_CONFIANCA = 0.95
SIGNIFICANCIA = 1 - NIVEL_CONFIANCA

## Problema 1

> Uma notícia sobre a Ilum que saiu recentemente no jornal Últimas Notícias dizia que os estudantes desta instituição são muito dedicados e leem em torno de 10 livros por ano. O que você pode dizer sobre essa afirmação?

Nesse contexto, nossa hipótese nula basicamente afirma que a média de livros lidos por ano é 10. Assim, vamos definir as hipóteses como sendo 

-   $H_0$: $\mu = 10$


-   $H_1$: $\mu \neq 10 $

### Tratamento de dados

Vamos excluir dos dados exemplos que possuam um número inválido de livros lidos.

In [4]:
df = df_original.copy()

quantos_livros = "Quantos livros você leu no ano passado?"
df = df.dropna(subset=quantos_livros, axis=0)


A depender da normalidade da amostra, podemos utilizar testes paramétricos, que tendem a oferecer resultados mais robustos. Verifquemos essa propriedade a partir do teste de Shapiro-Wilk:

In [5]:
pvalor = stats.shapiro(df[quantos_livros]).pvalue;
conclusao(pvalor, SIGNIFICANCIA)
print()

p-valor = 7.655347245597689e-17

Conclusão do teste de hipótese: devemos rejeitar H0



Visto que a normalidade da amostra não pode ser garantida, vamos aplicar o teste de Wilcoxon para verificar a $H_0$ de nosso problema.

In [6]:
MEDIA_HIPOTETICA = 10

diferencas = [livros_lidos - MEDIA_HIPOTETICA for livros_lidos in df[quantos_livros]]

pvalor = stats.wilcoxon(diferencas).pvalue

conclusao(pvalor, SIGNIFICANCIA)

p-valor = 0.027618480926544803

Conclusão do teste de hipótese: devemos rejeitar H0


Desse modo, não há evidência que suportem que os alunos tenham lido em média 10 livros no ano anterior.

## Problema 2

> O podcast sobre ciência Devaneios Probabilísticos discutiu recentemente um artigo que dizia que pessoas cujo número aleatório  escolhido é menor que 50 tendem a dormir mais horas por noite do que pessoas cujo número  aleatório escolhido é maior ou igual a 50. O que você pode dizer sobre essa afirmação considerando os dados do Censo Ilum²

Seja $n$ o número aleatório escolhido por uma pessoa e $h$ a média de horas que ela dorme por noite.

Nesta questão, vamos definir nossas hipóteses como

-   $H_0$: Quem escolheu $n \geq 50$ tem $h$ maior do que daqueles que escolheram $n < 50$.

-   $H_1$: Quem escolheu $n \geq 50$ tem $h$ menor ou igual daqueles que escolheram $n < 50$.

### Tratamento de dados

In [7]:
df = df_original.copy()

df.rename(columns={"Escolha um número aleatório entre 0 e 100": "num_aleatorio",
                    "Quantas horas (em média) você dorme por noite?": "media_sono"}, 
                    inplace=True)

# Não existe uma 25ª hora do dia!
logica1 = df["media_sono"] <= 24
df = df[logica1]

# Apenas números entre 0 e 100!
logica2 = df["num_aleatorio"].between(0, 100)
df = df[logica2]

# Exclui exemplos com dados faltantes para as duas variáveis de interesse
variaveis_de_interesse = ("num_aleatorio", "media_sono")
df = df.dropna(subset=variaveis_de_interesse, axis=0);

Separando dados de acordo com número escolhido

In [8]:
logica1 = df["num_aleatorio"] < 50
logica2 = df["num_aleatorio"] >= 50

menor_50 = df[logica1]
maior_50 = df[logica2]

sono_menor_50 = menor_50["media_sono"]
sono_maior_50 = maior_50["media_sono"]

Para esse tipo de situação em que devemos ocmparar médias de amostras, teste $t$ tendem a ser preferíveis, mas o tipo exato depende da natureza dos nosso dados. Vamos presumir que, no geral, a média das horas de sono de cada estudante independe daquela de outros estudantes. 

Vamos analisar quanto à normalidade da dsitribuição de cada grupo de médias de sono por meio do teste de Shapiro-Wilk:

In [9]:
medias = (sono_menor_50, sono_maior_50)
for media in medias:
    pvalor = stats.shapiro(media).pvalue
    conclusao(pvalor, SIGNIFICANCIA)
    print()

p-valor = 0.0034903234831521654

Conclusão do teste de hipótese: devemos rejeitar H0

p-valor = 1.8486076108486682e-05

Conclusão do teste de hipótese: devemos rejeitar H0



Agora a rejeição de $H_0$ do teste de Shapiro-Wilk nos inclina a utilizar o teste de Levene para avaliar a homoscedasticidade entre esses grupos:

In [10]:
pvalor = stats.levene(*medias).pvalue
conclusao(pvalor, SIGNIFICANCIA)

p-valor = 0.37988877597740617

Conclusão do teste de hipótese: não devemos rejeitar H0


Certo, então uma vez que os valores são independentes e temos que a homoscedasticidade entre as amostras não pode ser rejeitada, podemos utilizar o teste paramétrico $t$ para amostras independentes.

In [11]:
pvalor = stats.ttest_ind(
    sono_maior_50, sono_menor_50, 
    alternative="greater"
    ).pvalue
conclusao(pvalor, SIGNIFICANCIA)

p-valor = 0.6517053883613566

Conclusão do teste de hipótese: não devemos rejeitar H0


Desse modo, não há evidências o bastante para rejeitar a afirmação de $H_0$! Que curioso...

## Problema 3

> Recentemente viralizou no Y a notícia de que a altura média de quem prefere as cores Azul, Roxo, Rosa, Verde e Preto é igual. O que você pode dizer sobre essa afirmação considerando os dados do Censo Ilum?



Para esse problema, podemos definir nossas hipóteses como

-   $H_0$: A média de altura é igual entre pessoas com alguma dessas cores como favorita.

-   $H_1$: Há pelo menos uma das médias que é diferente das outras.

### Tratamento de dados

Excluindo exemplos em que altura ou cor favorita estão ausentes.

In [12]:
df = df_original.copy()
# Renomeando as colunas para facilitar leitura
df.rename(columns={'Qual é a sua altura em centímetros?': 'altura_cm',
                    "Das cores abaixo, qual é a sua favorita (ou a cor que mais gosta se sua cor favorita não estiver na lista)?": "cor_fav"}, 
                    inplace=True)

variaveis_de_interesse = ("altura_cm", "cor_fav")

df = df.dropna(axis=0, subset=variaveis_de_interesse)

Excluindo valores anômalos de altura e ajustando erros de notação ("," e ".")

In [13]:
# Filtrando valores anômalos de altura (A pessoa mais alta já registrada media 2.74 metros!)
df = df[df["altura_cm"] <= 274]

# Filtrando valores anômalos de altura (A menor pessoa adulta já registrada media 55 cm!)
df = df[df["altura_cm"] >= 55]

# Para corrigir erros de escala como "1,71" -> "171"
df["altura_cm"] = df["altura_cm"].replace(",", "")

In [14]:
altura_de_cores = {}

cores = ("Azul", "Roxo", "Rosa", "Verde", "Preto")
for cor in cores:
    logica = df["cor_fav"] == cor
    altura_de_cores[cor] = df[logica]["altura_cm"]

Visualizando altura média para os grupos de cores favoritas

In [15]:
print("Altura média em cm de quem tem por favorita a cor")
for cor in cores:
    print(f"    {cor}:  {np.mean(altura_de_cores[cor]):.2f}")

Altura média em cm de quem tem por favorita a cor
    Azul:  170.60
    Roxo:  170.13
    Rosa:  165.38
    Verde:  164.77
    Preto:  172.50


### Resolução

Para esse problema, podemos usar o teste ANOVA paramétrico, cujas suposições relevantes, conforme [1], são:

-   As variáveis aleatórias $X_1$, $X_2$, &hellip;, $X_k$ são independentes

O que de fato ocorre, uma vez que a altura de um aluno não irá interferir na altura de outro.

-   As variáveis aleatórias $X_1$, $X_2$, &hellip;, $X_k$ seguem distribuições normais (segundo a Ref. [5], podemos utilizar este teste mesmo que esta condição *não* seja verdadeira, principalmente em conjuntos de dados grandes)

Para verificar a normalidade das distribuições, utilizemos o teste de Shapiro-Wilk sobre cada conjunto de alturas, cuja hipótese nula afirma que a amostra tem por fonte uma população que segue uma distribuição normal.

In [16]:
for cor in cores:
    pvalor_shapiro = stats.shapiro(altura_de_cores[cor]).pvalue;
    conclusao(pvalor_shapiro, SIGNIFICANCIA)
    print()

p-valor = 0.21221803632130243

Conclusão do teste de hipótese: não devemos rejeitar H0

p-valor = 0.5208024201162363

Conclusão do teste de hipótese: não devemos rejeitar H0

p-valor = 0.5213944962370072

Conclusão do teste de hipótese: não devemos rejeitar H0

p-valor = 0.3392029749605703

Conclusão do teste de hipótese: não devemos rejeitar H0

p-valor = 0.0935251739953818

Conclusão do teste de hipótese: não devemos rejeitar H0



-   As variâncias das populações de onde as variáveis aleatórias $X_1$, $X_2$, &hellip;, $X_k$ foram amostradas são iguais. Esta condição tem um nome: homoscedasticidade.

Isso pode ser verificado por meio do teste de Bartlett, cuja $H_0$ afirma que todas as amostras originam de populações com a mesma variância. Esse teste funciona melhor com dados cuja população de origem siga uma distribuição normal, que como vimos, tem suporte de nosso teste anterior.

In [17]:
stat, pvalor = stats.bartlett(*altura_de_cores.values())
conclusao(pvalor, SIGNIFICANCIA)
print()

p-valor = 0.4494910452901839

Conclusão do teste de hipótese: não devemos rejeitar H0



Tendo cumprido as suposições relevantes do teste ANOVA, vamos aplicá-lo.

In [18]:
stat, pvalor = stats.f_oneway(*altura_de_cores.values())
conclusao(pvalor, SIGNIFICANCIA)
print()

p-valor = 0.13524949248563273

Conclusão do teste de hipótese: não devemos rejeitar H0



Portanto, não há evidências o suficiente para rejeitarmos a hipótese nula deste problema!

## Referências

[1] CASSAR, Daniel R. Material de aula **LMA-203 A.4 - Usando testes de hipóteses prontos usuais** para a disciplina de Estatística, 2026.

## Uso de IA neste trabalho

- Busca geral de testes adequados apresentados nos materiais da disciplina.

<div style="width: 100%; text-align: center; margin-bottom: 20px;">
  <img src="../elementos/rodape_branco.png" alt="Rodapé" style="max-width: 100%; height: auto;">
</div>